### Web Scraper Attempt 2

In [1]:
from bs4 import BeautifulSoup
import requests 
from selenium import webdriver
from selenium.webdriver.common.by import By
import io
from datetime import datetime as dt
from PIL import Image
import time
import os
import re

In [2]:
coluber = "https://www.google.com/search?q=coluber+constrictor&sca_esv=94226fca456b230b&sxsrf=ADLYWIJ040xDzhv73rD26UBZA3kjn7bM4Q:1730754443782&source=hp&biw=1440&bih=693&ei=izcpZ6SrLfnIkPIPwO_d0Qw&iflsig=AL9hbdgAAAAAZylFmzmv3GrhSu-Ew-TJ9wMmoR8hHVDJ&oq=coluber&gs_lp=EgNpbWciB2NvbHViZXIqAggAMgQQIxgnMgQQIxgnMgUQABiABDIFEAAYgAQyBRAAGIAEMgUQABiABDIFEAAYgAQyBRAAGIAEMgUQABiABDIFEAAYgARI8Q5QAFiDBXAAeACQAQCYATygAe4CqgEBN7gBAcgBAPgBAYoCC2d3cy13aXotaW1nmAIHoAKOA8ICCBAAGIAEGLEDwgILEAAYgAQYsQMYgwHCAg4QABiABBixAxiDARiKBcICDRAAGIAEGLEDGIMBGArCAgoQABiABBixAxgKwgIHEAAYgAQYCpgDAJIHATegB-Y3&sclient=img&udm=2#vhid=tmB2B-7MGPk3UM&vssid=mosaic"
blue_racer = "https://www.google.com/search?q=coluber+constrictor+foxii&sca_esv=b3aa834d5fa37549&sxsrf=ADLYWII9h-rU1AaaHyRq6f4ozhGWJp4BNg:1731809540676&source=hp&biw=1420&bih=722&ei=BFE5Z5O2JsGsur8PpNjNmAM&iflsig=AL9hbdgAAAAAZzlfFJsaUhY01Ey6C91aL2HTIYLCiAB9&ved=0ahUKEwiTyPOjpeKJAxVBlu4BHSRsEzMQ4dUDCBA&uact=5&oq=coluber+constrictor+foxii&gs_lp=EgNpbWciGWNvbHViZXIgY29uc3RyaWN0b3IgZm94aWkyBBAjGCcyBRAAGIAEMgUQABiABDIFEAAYgAQyBRAAGIAEMgUQABiABDIEEAAYHjIEEAAYHjIEEAAYHjIEEAAYHkjpAlAAWABwAHgAkAEAmAEnoAEnqgEBMbgBA8gBAPgBAvgBAYoCC2d3cy13aXotaW1nmAIBoAIvmAMAkgcBMaAHlAc&sclient=img&udm=2"
# Need to automate search queries

def get_images(delay, max, url):
    driver = webdriver.Chrome()
    def scroll_down(driver):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(delay)
        
    driver.get(url)

    image_urls = set()
    skips = 0
    thumbs = 0
    
    while len(image_urls) < max:
        print(f"Processed {thumbs} images so far.")
        curr_height = driver.execute_script("return document.body.scrollHeight")
        scroll_down(driver)
        if (driver.execute_script("return document.body.scrollHeight") == curr_height): # Reached bottom
            break
            
        thumbnails = driver.find_elements(By.CLASS_NAME, 'mNsIhb')
        for img in thumbnails[thumbs:max]:
            thumbs += 1
            try:
                img.click()
                time.sleep(delay)
            except:
                continue
            
            containers = driver.find_elements(By.CLASS_NAME, "YsLeY")
            for container in containers:                
                images = container.find_elements(By.TAG_NAME, "img")
                for image in images:
                    src = image.get_attribute('src')
                    if src and (re.search(r"\..{3}$", src) or re.search(r"\..{4}$", src)) and (src not in image_urls):
                        image_urls.add(src)
                    else:
                        max += 1
                        skips += 1
    print(f"There are now {len(image_urls)} unique images.")
    driver.quit()
    return image_urls

In [15]:
def download_img(download_path, url, file_name):
    try:
        headers = {
            "Accept": "image/*",
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
            "Referer": "https://www.researchgate.net/"
        }
        content = requests.get(url, headers = headers)
        content.raise_for_status()
        file = io.BytesIO(content.content)
        image = Image.open(file)
        

        #image.show()
        path = download_path + file_name + ".jpeg"
        with open(path, "wb") as f:
            try:
                image.save(f, "JPEG")
            except:
                image = image.convert('RGB')
                image.save(f, "JPEG")
    except Exception as e:
        print("Failed -", e)